In [2]:
import pandas as pd
import numpy as np
from pyprojroot import here

In [3]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"

In [4]:
df = pd.read_csv(RAW_DATA_DIR/"crime/Municipal-Delitos-2015-2025_abr2026.csv", encoding='latin-1')
pop = pd.read_csv(INTERIM_DATA_DIR/"population_2015_2025B.csv", encoding='latin-1')

In [5]:
df['CVEGEO'] = df['Cve. Municipio'].astype(str).str.zfill(5)
df['crime_count'] = df.iloc[:, 9:21].sum(axis=1)
df = df.rename(columns={'Entidad': 'state', 'Municipio': 'municipality', 'Año': 'year'})

In [6]:
df['cvemun'] = df['Cve. Municipio'] - df['Clave_Ent'] * 1000
rows_to_drop = df[df['cvemun'] > 600].index
df = df.drop(index=rows_to_drop)

In [7]:
df['CVEGEO'].unique()

array(['01001', '01002', '01003', ..., '24059', '25019', '25020'],
      shape=(2478,), dtype=object)

In [8]:
A_crime = [ # Criminal activity associated with organized crime by the Federal Law Against Organized Crime
    'Secuestro',
    'Trata de personas',
    'Narcomenudeo',
    'Falsificación',
    'Corrupción de menores'
]

In [9]:
crime_df = df[df['Tipo de delito'].isin(A_crime)].copy()
A_crime_df = crime_df.groupby(['year', 'CVEGEO', 'state', 'municipality'])['crime_count'].sum().drop(columns='Unnamed: 0').reset_index()

In [10]:
munis_2025 = df[df['year'] == 2025][['CVEGEO']].drop_duplicates()

years = df['year'].unique()
skeleton = munis_2025.assign(key=1).merge(
    pd.DataFrame({'year': years, 'key': 1}), on='key'
).drop(columns='key')

ACdf_fixed = skeleton.merge(A_crime_df, on=['CVEGEO', 'year'], how='left')
ACdf_fixed['crime_count'] = ACdf_fixed['crime_count'].fillna(0).astype(int)
ACdf_fixed = ACdf_fixed.sort_values(['year', 'CVEGEO']).reset_index(drop=True)

In [11]:
ACdf_fixed.to_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding="latin-1")

ACdf_fixed.sample(5)

,CVEGEO,year,state,municipality,crime_count
10807,16101,2019,Michoacán de Ocampo,Tzitzio,1
10955,20029,2019,Oaxaca,Eloxochitlán de Flores Magón,0
769,15100,2015,México,Tezoyuca,2
6074,20104,2017,Oaxaca,San Antonino el Alto,0
21991,30066,2023,Veracruz de Ignacio de la Llave,Espinal,5


In [12]:
ACdf_fixed["CVEGEO"] = pd.to_numeric(ACdf_fixed["CVEGEO"], errors="coerce").astype("Int64").astype(str)
pop["CVEGEO"] = pd.to_numeric(pop["CVEGEO"], errors="coerce").astype("Int64").astype(str)
# pop_long["CVEGEO"] = pd.to_numeric(pop["CVEGEO"], errors="coerce").astype("Int64").astype(str)

year_cols = [c for c in pop.columns if c.isdigit() and len(c) == 4]

pop_long = pop.melt(
    id_vars=["CVEGEO"],
    value_vars=year_cols,
    var_name="year",
    value_name="population"
)

pop_long["year"] = pop_long["year"].astype(int)
ACdf_fixed["year"] = ACdf_fixed["year"].astype(int)

merged = ACdf_fixed.merge(
    pop_long,
    on=["CVEGEO", "year"],
    how="left"
)

In [15]:
merged['log_pop'] = np.log(merged['population'])
missing = merged["population"].isna().sum()
print(f"Rows with no matching population: {missing}")
# complete = merged.fillna(0)
# missing = complete["population"].isna().sum()
# print(f"Rows with no matching population: {missing}")

Rows with no matching population: 229
Rows with no matching population: 0


In [18]:
merged.to_csv(INTERIM_DATA_DIR/"crime_with_population.csv", index=False)